In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import time
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
print(f"Project root added to sys.path: {project_root}")

Project root added to sys.path: c:\Users\au584144\Repos\lab-controller


In [ ]:
import serial
import time

class FireStingLegacy:
    def __init__(self, port, baud=19200, timeout=1.0):
        self.port = port
        self.baud = baud
        self.timeout = timeout
        self.ser = None
    
    def connect(self):
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baud,
                parity=serial.PARITY_NONE,
                stopbits=serial.STOPBITS_ONE,
                bytesize=serial.EIGHTBITS,
                timeout=self.timeout
            )
            time.sleep(0.5)
            self.flush()
            print(f"Connected to {self.port} at {self.baud} baud.")
            return True
        except serial.SerialException as e:
            print(f"Connection failed: {e}")
            return False

    def flush(self):
        if self.ser and self.ser.is_open:
            self.ser.reset_input_buffer()
            self.ser.reset_output_buffer()

    def send_command_wait_long(self, cmd, wait_time=2.0):
        if not self.ser or not self.ser.is_open:
            return

        print(f"\n--- Sending: {cmd} ---")
        self.flush() 
        
        try:
            cmd_bytes = (cmd + '\r').encode('ascii')
            self.ser.write(cmd_bytes)
            
            start_time = time.time()
            full_response = b""
            
            while (time.time() - start_time) < wait_time:
                if self.ser.in_waiting > 0:
                    chunk = self.ser.read(self.ser.in_waiting)
                    if chunk:
                        full_response += chunk
                        # print(f"Received chunk: {chunk}") # Reduce noise
                time.sleep(0.1)
            
            print(f"Full Response (Raw): {full_response}")
            try:
                decoded = full_response.decode('ascii').strip()
                if decoded:
                    print(f"Full Response (Decoded): {decoded}")
            except:
                pass

        except Exception as e:
            print(f"Error: {e}")

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()

Connected to COM10 at 19200 baud.

--- Sending: #MEA ---
Full Response (Raw): b''

--- Sending: #MSR ---
Full Response (Raw): b''

--- Sending: #VAL ---
Full Response (Raw): b''

--- Sending: #GET ---
Full Response (Raw): b''

--- Sending: #READ ---
Full Response (Raw): b''

--- Sending: #TMP ---
Full Response (Raw): b''

--- Sending: #IDNR ---
Full Response (Raw): b''

--- Sending: #LOGO ---
Full Response (Raw): b'#LOGO\r'
Full Response (Decoded): #LOGO

--- Sending: #PARA ---
Full Response (Raw): b''

--- Sending: #CONF ---
Full Response (Raw): b''

--- Sending: MEA ---
Full Response (Raw): b''

--- Sending: MSR ---
Full Response (Raw): b''

--- Sending: READ ---
Full Response (Raw): b''

--- Sending: VAL ---
Full Response (Raw): b''

--- Sending: TMP ---
Full Response (Raw): b''

--- Sending: GET ---
Full Response (Raw): b''

--- Sending: MEA 0 ---
Full Response (Raw): b''

--- Sending: MSR 0 ---
Full Response (Raw): b''

--- Sending: READ 0 ---
Full Response (Raw): b''

--- Sending

In [4]:
# --- Test ---
fs = FireStingLegacy('COM10', 19200)
if fs.connect():
    try:
        # 1. Try commands with '#' prefix (since #VERS worked)
        commands_hash = ["#MEA", "#MSR", "#VAL", "#GET", "#READ", "#TMP", "#IDNR", "#LOGO", "#PARA", "#CONF"]
        
        # 2. Try commands without arguments (if single channel)
        commands_no_arg = ["MEA", "MSR", "READ", "VAL", "TMP", "GET"]
        
        # 3. Try commands with channel 0 (sometimes 0-indexed)
        commands_ch0 = ["MEA 1", "MSR 1", "READ 1", "VAL 1", "TMP 1"]

        all_commands = commands_hash + commands_no_arg + commands_ch0
        
        for cmd in all_commands:
            fs.send_command_wait_long(cmd, wait_time=1.0)
    finally:
        fs.close()

Connected to COM10 at 19200 baud.

--- Sending: #MEA ---
Full Response (Raw): b''

--- Sending: #MSR ---
Full Response (Raw): b''

--- Sending: #VAL ---
Full Response (Raw): b''

--- Sending: #GET ---
Full Response (Raw): b''

--- Sending: #READ ---
Full Response (Raw): b''

--- Sending: #TMP ---
Full Response (Raw): b''

--- Sending: #IDNR ---
Full Response (Raw): b''

--- Sending: #LOGO ---
Full Response (Raw): b'#LOGO\r'
Full Response (Decoded): #LOGO

--- Sending: #PARA ---
Full Response (Raw): b''

--- Sending: #CONF ---
Full Response (Raw): b''

--- Sending: MEA ---
Full Response (Raw): b''

--- Sending: MSR ---
Full Response (Raw): b''

--- Sending: READ ---
Full Response (Raw): b''

--- Sending: VAL ---
Full Response (Raw): b''

--- Sending: TMP ---
Full Response (Raw): b''

--- Sending: GET ---
Full Response (Raw): b''

--- Sending: MEA 1 ---
Full Response (Raw): b''

--- Sending: MSR 1 ---
Full Response (Raw): b'MSR 1\r'
Full Response (Decoded): MSR 1

--- Sending: READ 1 ---